In [2]:
# Import libraries
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

# Load dataset
data = pd.read_csv(r"C:\Users\LENOVO\Downloads\student_logistic_dataset.csv")

# Separate features and target
X = data.drop("result", axis=1)
y = data["result"]

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Feature scaling
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Create Logistic Regression model
model = LogisticRegression(max_iter=1000)

# Train model
model.fit(X_train, y_train)

# Predictions
y_pred = model.predict(X_test)

# Model evaluation
print("Accuracy:", round(accuracy_score(y_test, y_pred)*100,2), "%")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))


# Predict for a new student
new_student = [[
    90,     # attendance_percentage
    0,     # internal_marks
    80,     # assignment_completion_rate
    56,     # study_hours_per_week
    0,      # extracurricular_participation
    9.5,    # previous_gpa
    78,     # class_participation_score
    7.5,    # sleep_duration
    1       # internet_usage_for_study
]]

# Apply same scaling
new_student_scaled = scaler.transform(new_student)

prediction = model.predict(new_student_scaled)

# Display result
if prediction[0] == 0:
    print("\nPrediction: Student likely to PASS")
else:
    print("\nPrediction: Student likely to FAIL / At Risk")

Accuracy: 80.0 %

Confusion Matrix:
[[38 12]
 [ 8 42]]

Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.76      0.79        50
           1       0.78      0.84      0.81        50

    accuracy                           0.80       100
   macro avg       0.80      0.80      0.80       100
weighted avg       0.80      0.80      0.80       100


Prediction: Student likely to FAIL / At Risk


C:\Users\LENOVO\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [1]:
# =============================================================
#  Basic ML Model — Loan Default Prediction
#  Algorithm: Logistic Regression + Random Forest
# =============================================================

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

# ─────────────────────────────────────────────
# 1. Create Sample Data
# ─────────────────────────────────────────────
np.random.seed(42)
n = 1000

df = pd.DataFrame({
    "age":           np.random.randint(22, 60, n),
    "income":        np.random.randint(20000, 120000, n),
    "credit_score":  np.random.randint(300, 850, n),
    "loan_amount":   np.random.randint(5000, 80000, n),
    "employment":    np.random.choice(["Salaried", "Self-Employed", "Unemployed"], n),
})

# Target: 1 = default, 0 = no default
df["default"] = (
    (df["credit_score"] < 550).astype(int) +
    (df["income"] < 35000).astype(int) +
    (df["employment"] == "Unemployed").astype(int)
) >= 2
df["default"] = df["default"].astype(int)

print("Dataset:\n", df.head())
print(f"\nDefault rate: {df['default'].mean():.2%}")

# ─────────────────────────────────────────────
# 2. Preprocess
# ─────────────────────────────────────────────
df["employment"] = LabelEncoder().fit_transform(df["employment"])

X = df.drop("default", axis=1)
y = df["default"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

# ─────────────────────────────────────────────
# 3. Train Models
# ─────────────────────────────────────────────

# Logistic Regression
lr = LogisticRegression()
lr.fit(X_train, y_train)
lr_preds = lr.predict(X_test)

# Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
rf_preds = rf.predict(X_test)

# ─────────────────────────────────────────────
# 4. Evaluate
# ─────────────────────────────────────────────
for name, preds, model in [("Logistic Regression", lr_preds, lr), ("Random Forest", rf_preds, rf)]:
    print(f"\n{'='*40}")
    print(f"  {name}")
    print(f"{'='*40}")
    print(f"Accuracy : {accuracy_score(y_test, preds):.4f}")
    print(f"ROC-AUC  : {roc_auc_score(y_test, model.predict_proba(X_test)[:,1]):.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, preds, target_names=["No Default", "Default"]))

# ─────────────────────────────────────────────
# 5. Predict on New Data
# ─────────────────────────────────────────────
new_customer = pd.DataFrame([{
    "age": 35, "income": 45000, "credit_score": 520,
    "loan_amount": 20000, "employment": 0  # 0 = Salaried (after encoding)
}])
new_customer_scaled = scaler.transform(new_customer)
prediction = rf.predict(new_customer_scaled)[0]
probability = rf.predict_proba(new_customer_scaled)[0][1]

print("\n" + "="*40)
print("  New Customer Prediction")
print("="*40)
print(f"Prediction  : {'DEFAULT' if prediction == 1 else 'NO DEFAULT'}")
print(f"Default Prob: {probability:.2%}")

Dataset:
    age  income  credit_score  loan_amount     employment  default
0   50   22717           350        56098  Self-Employed        1
1   36   79676           623        67154  Self-Employed        0
2   29   47952           440        17991       Salaried        0
3   42  106444           769        73344     Unemployed        0
4   40   62460           633        41593     Unemployed        0

Default rate: 21.40%

  Logistic Regression
Accuracy : 0.9100
ROC-AUC  : 0.9507

Classification Report:
              precision    recall  f1-score   support

  No Default       0.91      0.97      0.94       152
     Default       0.89      0.71      0.79        48

    accuracy                           0.91       200
   macro avg       0.90      0.84      0.87       200
weighted avg       0.91      0.91      0.91       200


  Random Forest
Accuracy : 1.0000
ROC-AUC  : 1.0000

Classification Report:
              precision    recall  f1-score   support

  No Default       1.00      1